# Modul 3 — Operasi Citra Sederhana

**Pengolahan Citra dan Visi Komputer — Jurusan Teknologi Informasi**

Notebook ini merupakan pengerjaan lengkap Modul 3 pada lingkungan **Kaggle** (tanpa
`google.colab`, tanpa `drive.mount`, tanpa `files.upload()`). Seluruh berkas citra
diakses melalui variabel `BASE`/`WEEK` yang otomatis menunjuk ke
`/kaggle/input/pcvk-images/week3` (Kaggle) atau folder `datasets/week3` (lokal).

Cakupan notebook:

| Bagian | Isi |
|---|---|
| D1 | Inverse citra, transformasi linier brightness (truncate), transformasi contrast, transformasi logarithmic brightness |
| Tugas 1–4 | Inverse, contrast, log brightness, grayscale (averaging / lightness / luminance) |
| Tugas 5–7 | Selective color (merah dipertahankan), gamma correction (input pengguna), simulasi bit depth 1–7 |
| Tugas 8–9 | Average denoising + PSNR, image masking (NOT/OR/AND/NAND/XOR) |
| Tugas 10–11 | Foto malam hari, perbaikan kualitas `crayfish.jpg` |

Seluruh operasi dasar (inverse, brightness, contrast, log, grayscale, gamma, bit depth)
diimplementasikan **manual dengan NumPy** mengikuti formula pada Ulasan Teori — bukan
memakai fungsi siap pakai seperti `cv.convertScaleAbs`, `cv.cvtColor`, `cv.threshold`,
atau `cv.LUT`. Fungsi *truncate* diimplementasikan dengan `np.clip`.

In [ ]:
# ==== KONFIGURASI LINGKUNGAN (jalan di Kaggle maupun lokal) ====
import os, sys, glob, math
import numpy as np
import cv2 as cv
import matplotlib.pyplot as plt

def _find_base():
    for p in ('/kaggle/input/pcvk-images',
              '/kaggle/input/datasets/singgihwahyupermana/pcvk-images'):
        if os.path.isdir(p):
            return p
    _hit = glob.glob('/kaggle/input/**/pcvk-images', recursive=True)
    if _hit:
        return _hit[0]
    d = os.getcwd()
    for _ in range(6):
        if all(os.path.isdir(os.path.join(d, f'week{i}')) for i in (1, 2, 3)):
            return d
        d = os.path.dirname(d)
    return None

BASE = _find_base()
WEEK = os.path.join(BASE, 'week3', 'assets')   # <-- ganti angka minggu sesuai notebook

def show(img, title='', cmap=None, figsize=(6, 4)):
    plt.figure(figsize=figsize)
    if img.ndim == 3:
        plt.imshow(cv.cvtColor(img, cv.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap=cmap or 'gray')
    plt.title(title); plt.axis('off'); plt.show()

# Shim cv2_imshow (Colab) -> Kaggle
try:
    from google.colab.patches import cv2_imshow
except Exception:
    def cv2_imshow(img):
        show(img)

print('BASE :', BASE)
print('WEEK :', WEEK)
print('files:', sorted(os.listdir(WEEK)) if WEEK and os.path.isdir(WEEK) else 'MISSING')

## D1. Operasi Citra Sederhana

Modul asli melakukan *mount* Google Drive pada langkah ini. Pada versi Kaggle, langkah
tersebut digantikan oleh sel konfigurasi di atas (`BASE`/`WEEK`), sehingga tidak ada
`drive.mount` maupun otorisasi Google yang diperlukan. Seluruh berkas minggu ini berada di
`WEEK` (`peppers.jpg`, `galaxy.jpg`, `crayfish.jpg`, `couple.tiff`, dan folder
`noises/` berisi 100 citra ber-noise).

### C1(a) — Inverse Citra

Citra negatif diperoleh dengan mencerminkan nilai pixel terhadap nilai tengah 255:

$$g(x) = 255 - f(x)$$

Jika nilai pixel asli 255 maka hasilnya 0, dan sebaliknya. Operasi ini murni aritmatika
per-pixel sehingga tidak memerlukan fungsi library apa pun.

In [ ]:
# ---- Inverse Citra : g(x) = 255 - f(x) ----
img = cv.imread(os.path.join(WEEK, 'peppers.jpg'))
assert img is not None, 'peppers.jpg tidak ditemukan di ' + str(WEEK)

inverse = 255 - img          # implementasi manual (aritmatika NumPy), tanpa cv.bitwise_not

show(img, 'Citra Asli - peppers.jpg')
show(inverse, 'Inverse Citra (citra negatif)')

print('nilai pixel minimum/maksimum citra asli :', img.min(), '/', img.max())
print('nilai pixel minimum/maksimum citra negatif:', inverse.min(), '/', inverse.max())

### C1(b) — Transformasi Linier Brightness

$$g(x,y) = f(x,y) + b$$

Operasi ini satu arah (*irreversible*): karena hasilnya harus tetap berada pada rentang
0–255, diterapkan fungsi **truncate** — nilai > 255 dipaksa menjadi 255 dan nilai < 0
dipaksa menjadi 0. Akibatnya informasi yang hilang tidak dapat dikembalikan.

Algoritma truncate: (1) baca nilai pixel $f_1$; (2) jika $f_1 < 0$ → 0; (3) jika
$f_1 > 255$ → 255.

In [ ]:
# ---- Fungsi truncate manual (np.clip) ----
def truncate(arr):
    """Truncate: nilai < 0 -> 0, nilai > 255 -> 255 (output tetap uint8)."""
    return np.clip(arr, 0, 255).astype(np.uint8)

# ---- Transformasi Linier Brightness : g(x,y) = f(x,y) + b ----
def linear_brightness(image, b):
    return truncate(image.astype(np.float64) + b)

for b in (-80, 80, 150):
    hasil = linear_brightness(img, b)
    show(hasil, 'Linear Brightness  b = %+d' % b)
    print('b = %+4d -> rata-rata intensitas %.2f (asli %.2f)' % (b, hasil.mean(), img.mean()))

# Bukti sifat irreversible: 240 + 20 -> 255, lalu 255 - 20 -> 235 (bukan 240)
demo = np.array([[[240, 240, 240]]], dtype=np.uint8)
naik = linear_brightness(demo, 20)
turun = linear_brightness(naik, -20)
print('demo pixel 240 -> +20 =', naik[0, 0, 0], '-> -20 =', turun[0, 0, 0], '(tidak kembali ke 240)')

### C1(c) — Transformasi Contrast

Selain bentuk $g(x,y) = a \cdot f(x,y) + b$, kontras dapat dihitung dengan
**contrast correction factor**:

$$F = \frac{259\,(C + 255)}{255\,(259 - C)}$$

dengan $C$ = level kontras. Nilai $F$ **harus bertipe double (float)**, bukan integer.
Setiap kanal R, G, B dihitung:

$$R' = F(R - 128) + 128, \qquad G' = F(G - 128) + 128, \qquad B' = F(B - 128) + 128$$

Hasilnya dapat keluar dari rentang 0–255 sehingga wajib di-*truncate* (konsekuensinya
operasi ini juga tidak dapat direverse).

In [ ]:
# ---- Transformasi Contrast dengan Contrast Correction Factor ----
def contrast_transform(image, C):
    """F = 259(C+255) / (255(259-C))  -> bertipe double; lalu R'=F(R-128)+128 + truncate."""
    F = (259.0 * (C + 255.0)) / (255.0 * (259.0 - C))
    out = F * (image.astype(np.float64) - 128.0) + 128.0
    return truncate(out)

for Cval in (-60, 60, 120):
    hasil = contrast_transform(img, Cval)
    show(hasil, 'Contrast  C = %+d  (F = %.3f)' % (Cval, (259.0*(Cval+255.0))/(255.0*(259.0-Cval))))
    print('C = %+4d -> std intensitas %.2f (asli %.2f)' % (Cval, hasil.std(), img.std()))

### C1(d) — Transformasi Logarithmic Brightness

$$s = c \cdot \log(1 + r)$$

Transformasi log memetakan rentang gray-level input yang sempit (terutama nilai rendah)
menjadi rentang output yang lebih lebar, sehingga detail pada area gelap menjadi terlihat.
Konstanta $c$ dipilih agar hasilnya memenuhi skala 0–255:
$c = \dfrac{255}{\log(1 + r_{max})}$.

In [ ]:
# ---- Transformasi Logarithmic Brightness : s = c * log(1 + r) ----
def log_brightness(image, c=None):
    f = image.astype(np.float64)
    if c is None:
        c = 255.0 / np.log(1.0 + f.max())   # normalisasi ke skala 0-255
    return truncate(c * np.log(1.0 + f))

show(img, 'Citra Asli - peppers.jpg')
show(log_brightness(img), 'Logarithmic Brightness (c = 255/log(1+rmax))')
show(log_brightness(img, c=30.0), 'Logarithmic Brightness (c = 30)')
print('rata-rata intensitas asli        : %.2f' % img.mean())
print('rata-rata intensitas log (auto c): %.2f' % log_brightness(img).mean())

### C1(e) — Grayscale

Tiga metode grayscaling yang diimplementasikan manual:

$$Gray_{Averaging} = \frac{R + G + B}{3}$$

$$Gray_{Lightness} = \frac{\max[R,G,B] + \min[R,G,B]}{2}$$

$$Gray_{Luminance} = 0.21R + 0.72G + 0.07B$$

Luminance memberi bobot sesuai sensitivitas spektrum visible (mata paling sensitif pada
hijau), sehingga paling mendekati persepsi mata manusia. Perhatikan bahwa OpenCV membaca
citra dalam urutan kanal **BGR**, sehingga perlu pemetaan indeks yang benar.

In [ ]:
# ---- Tiga metode grayscale (manual) ----
def gray_averaging(image):
    f = image.astype(np.float64)
    return truncate((f[:, :, 0] + f[:, :, 1] + f[:, :, 2]) / 3.0)   # B,G,R

def gray_lightness(image):
    mx = image.max(axis=2).astype(np.float64)
    mn = image.min(axis=2).astype(np.float64)
    return truncate((mx + mn) / 2.0)

def gray_luminance(image):
    B, G, R = (image[:, :, i].astype(np.float64) for i in range(3))
    return truncate(0.21 * R + 0.72 * G + 0.07 * B)

show(img, 'Citra Asli - peppers.jpg')
show(gray_averaging(img), 'Grayscale - Averaging  (R+G+B)/3')
show(gray_lightness(img), 'Grayscale - Lightness  (max+min)/2')
show(gray_luminance(img), 'Grayscale - Luminance  0.21R+0.72G+0.07B')

print('rata-rata  averaging : %.2f' % gray_averaging(img).mean())
print('rata-rata  lightness : %.2f' % gray_lightness(img).mean())
print('rata-rata  luminance : %.2f' % gray_luminance(img).mean())

### Tugas 5 — Selective Color (pertahankan merah, sisanya grayscale)

Pendekatan: bangun **mask merah** dari perbandingan kanal — sebuah pixel dianggap merah
jika kanal R lebih dominan daripada G dan B secara signifikan. Pixel di dalam mask
dipertahankan warna aslinya, sedangkan pixel di luar mask diganti dengan nilai luminance
(grayscale).

In [ ]:
# ---- Selective color: warna merah dipertahankan, sisanya grayscale ----
def selective_red(image, delta=30):
    B, G, R = (image[:, :, i].astype(np.float64) for i in range(3))
    mask = (R > G) & (R > B) & ((R - np.maximum(G, B)) > delta)   # piksel dominan merah
    gray = gray_luminance(image)
    out = np.repeat(gray[:, :, None], 3, axis=2)                 # semua jadi grayscale
    out[mask] = image[mask]                                      # kembalikan warna merah
    return out.astype(np.uint8), mask

hasil_sel, mask_merah = selective_red(img, delta=30)
show(img, 'Citra Asli - peppers.jpg')
show(hasil_sel, 'Selective Color: merah dipertahankan, sisanya grayscale')
show((mask_merah * 255).astype(np.uint8), 'Mask piksel merah (putih = dipertahankan)')
print('persentase piksel merah yang dipertahankan: %.2f%%' % (100.0 * mask_merah.mean()))

### C2(a) — Gamma Correction

Rumus gamma correction pada modul:

$$I' = 255 \times \left(\frac{I}{255}\right)^{1/\gamma}$$

Nilai $\gamma$ diminta dari pengguna melalui `input()`. Karena `input()` **memblokir
eksekusi** dan tidak tersedia pada eksekusi otomatis (Kaggle *commit*/Run All), tersedia
variabel default `gamma_value` dan flag `USE_INTERACTIVE_INPUT`. Saat flag `False`,
notebook berjalan penuh tanpa pernah menunggu input pengguna.

Catatan penting: dengan rumus modul (eksponen $1/\gamma$), nilai $\gamma > 1$ **mencerahkan**
citra dan $\gamma < 1$ **menggelapkan** citra.

In [ ]:
# ---- Gamma Correction (input pengguna dengan fallback non-interaktif) ----
print(' Gamma Correction pada citra ')
print('----------------------------------')

USE_INTERACTIVE_INPUT = False   # True bila dijalankan interaktif (Colab/lokal); False di Kaggle
gamma_value = 3                 # fallback default agar tidak memblokir eksekusi

if USE_INTERACTIVE_INPUT:
    try:
        gamma_value = int(input('Masukkan nilai Gamma: '))
    except ValueError:
        print('Error, not a number')

print('Nilai Gamma yang dipakai:', gamma_value)

def gamma_correction(image, gamma):
    f = image.astype(np.float64) / 255.0
    return truncate(255.0 * np.power(f, 1.0 / gamma))

show(img, 'Citra Asli - peppers.jpg')
show(gamma_correction(img, gamma_value), 'Gamma Correction (gamma = %s, input pengguna)' % gamma_value)
show(gamma_correction(img, 0.5), 'Gamma Correction (gamma = 0.5) -> lebih gelap')
show(gamma_correction(img, 6.0), 'Gamma Correction (gamma = 6) -> lebih terang')
for g in (0.5, 3, 6):
    print('gamma = %-4s -> rata-rata intensitas %.2f' % (g, gamma_correction(img, g).mean()))

### C2(b) — Simulasi Image Depth (Kuantisasi)

$$level = \frac{255}{2^{bit\_depth} - 1}, \qquad C' = round\left(\frac{C}{level}\right) \times level$$

Citra 8-bit memiliki 256 level warna (0–255). Dengan `bit_depth = n`, jumlah level menjadi
$2^n$ sehingga gradasi warna menyusut — inilah simulasi kuantisasi. Contoh: `bit_depth = 4`
menghasilkan level 255/15 = 17, dengan gradasi 0, 17, 34, …, 255 (16 macam).

In [ ]:
# ---- Simulasi Image Depth (kuantisasi 1-7 bit) ----
def bit_depth_simulation(gray_image, bit_depth):
    level = 255.0 / (pow(2, bit_depth) - 1)
    return truncate(np.round(gray_image.astype(np.float64) / level) * level)

gray_peppers = gray_luminance(img)
show(gray_peppers, 'Citra Grayscale 8-bit (referensi)')

for bd in range(1, 8):
    hasil_bd = bit_depth_simulation(gray_peppers, bd)
    level = 255.0 / (pow(2, bd) - 1)
    show(hasil_bd, 'bit_depth = %d  (level = %.1f, %d macam warna)' % (bd, level, 2 ** bd))
    print('bit_depth = %d -> jumlah warna %3d, jumlah nilai unik pada citra: %d'
          % (bd, 2 ** bd, len(np.unique(hasil_bd))))

### C2(c)–(d) — PSNR dan Average Denoising

Average denoising menjumlahkan citra ber-noise yang berkoordinat sama lalu membaginya
dengan jumlah citra. Noise yang bersifat acak (tidak berkorelasi antar citra) saling
menghapus, sedangkan komponen sinyal yang tetap menguat. Semakin banyak citra dirata-rata,
semakin kecil noise tersisa sehingga **PSNR meningkat**.

$$MSE = \frac{1}{MN}\sum_{i}\sum_{j}(O(i,j) - D(i,j))^2, \qquad
PSNR = 10\log_{10}\left(\frac{(L-1)^2}{MSE}\right) = 20\log_{10}\left(\frac{L-1}{RMSE}\right)$$

dengan $L = 256$. Citra asli: `galaxy.jpg`; 100 citra ber-Gaussian-noise: `noises/*.jpg`.

In [ ]:
# ---- Fungsi PSNR (sesuai modul) ----
def PSNR(img1, img2):
    mse = np.mean((img1.astype(np.float64) - img2.astype(np.float64)) ** 2)
    if mse == 0:          # tidak ada noise sama sekali -> PSNR tidak bermakna
        return 100
    max_pixel = 255.0
    return 20 * math.log10(max_pixel / math.sqrt(mse))

# ---- Membaca citra asli dan 100 citra ber-noise ----
galaxy = cv.imread(os.path.join(WEEK, 'galaxy.jpg'))
assert galaxy is not None, 'galaxy.jpg tidak ditemukan'

cv_img = []
for path in sorted(glob.glob(os.path.join(WEEK, 'noises', '*.jpg'))):
    n = cv.imread(path)
    cv_img.append(n)

print('jumlah citra ber-noise yang dibaca:', len(cv_img))
show(galaxy, 'Citra Asli - galaxy.jpg')
show(cv_img[0], 'Contoh satu citra ber-noise (noises/noise_001.jpg)')
print('PSNR 1 citra ber-noise terhadap citra asli: %.4f dB' % PSNR(cv_img[0], galaxy))

In [ ]:
# ---- Average denoising untuk 10, 20, 40, 80, 100 citra ----
avg_results = {}
for jumlah in (10, 20, 40, 80, 100):
    akumulasi = np.zeros(galaxy.shape, dtype=np.float64)
    for im in cv_img[:jumlah]:
        akumulasi += im.astype(np.float64)
    hasil = truncate(akumulasi / jumlah)
    avg_results[jumlah] = hasil
    nilai = PSNR(hasil, galaxy)
    print('Average %3d citra -> PSNR = %.4f dB' % (jumlah, nilai))
    show(hasil, 'Average %d citra (PSNR = %.2f dB)' % (jumlah, nilai), figsize=(5, 4))

In [ ]:
# ---- Tabel ringkasan PSNR (untuk jawaban pada markdown di bawah) ----
print('| No | Jumlah Citra di Average | Nilai PSNR (dB) |')
print('|----|-------------------------|------------------|')
for i, jumlah in enumerate((10, 20, 40, 80, 100), start=1):
    print('| %d | %d | %.4f |' % (i, jumlah, PSNR(avg_results[jumlah], galaxy)))
print()
print('| Jumlah | PSNR (dB) | Selisih terhadap sebelumnya |')
print('|--------|-----------|------------------------------|')
prev = None
for jumlah in (10, 20, 40, 80, 100):
    nilai = PSNR(avg_results[jumlah], galaxy)
    print('| %d | %.4f | %s |' % (jumlah, nilai, '-' if prev is None else '%+.4f' % (nilai - prev)))
    prev = nilai

### C2(e) — Image Masking (Operator Logika)

Masking adalah operasi logika per-pixel. Untuk dua citra biner A dan B:

| Operator | Rumus | Makna |
|---|---|---|
| NOT | $\bar{A}$ | komplemen / kebalikan |
| OR | $A \cup B$ | pixel aktif bila A **atau** B aktif |
| AND | $A \cap B$ | pixel aktif bila A **dan** B aktif (irisan) |
| NAND | $\overline{A \cap B}$ | kebalikan dari AND |
| XOR | $A \oplus B$ | aktif bila hanya salah satu yang aktif |

Citra uji: `couple.tiff`. Citra dibinerkan menjadi citra biner **A** menggunakan threshold
Otsu yang dihitung manual (histogram + variansi antar-kelas, tanpa `cv.threshold`), lalu
**B** adalah A yang digeser 40 pixel ke kanan sehingga bagian yang overlap terlihat jelas
pada OR/AND/XOR. Operasi logika dilakukan langsung dengan operator NumPy (`&`, `|`, `^`,
`255 - x`).

In [ ]:
# ---- Membinerkan couple.tiff dengan threshold Otsu MANUAL ----
couple = cv.imread(os.path.join(WEEK, 'couple.tiff'))
assert couple is not None, 'couple.tiff tidak ditemukan'
gray_couple = gray_luminance(couple)

def otsu_threshold_manual(gray):
    """Threshold Otsu dihitung manual dari histogram (tanpa cv.threshold)."""
    hist = np.bincount(gray.ravel(), minlength=256).astype(np.float64)
    total = gray.size
    sum_total = float(np.dot(np.arange(256), hist))
    sumB = 0.0; wB = 0.0; best = -1.0; thr = 0
    for t in range(256):
        wB += hist[t]
        if wB == 0:
            continue
        wF = total - wB
        if wF == 0:
            break
        sumB += t * hist[t]
        mB = sumB / wB
        mF = (sum_total - sumB) / wF
        var_between = wB * wF * (mB - mF) ** 2
        if var_between > best:
            best = var_between; thr = t
    return thr

thr = otsu_threshold_manual(gray_couple)
print('threshold Otsu manual =', thr)

A = ((gray_couple > thr) * 255).astype(np.uint8)          # citra biner A (foreground objek terang)
B = np.roll(A, 40, axis=1)                                # citra biner B (A digeser 40 px ke kanan)
show(couple, 'Citra Asli - couple.tiff')
show(A, 'A: biner couple.tiff (threshold Otsu = %d)' % thr)
show(B, 'B: A digeser 40 piksel ke kanan')

In [ ]:
# ---- Operator logika: NOT, OR, AND, NAND, XOR ----
operasi = {
    'NOT (komplemen)': 255 - A,
    'OR (atau)':       A | B,
    'AND (dan)':       A & B,
    'NAND (not and)':  255 - (A & B),
    'XOR (exclusive or)': A ^ B,
}

for nama, hasil_op in operasi.items():
    show(hasil_op, 'Operator %s' % nama)
    print('%-20s -> persentase piksel putih: %6.2f%%' % (nama, 100.0 * (hasil_op > 127).mean()))

# Contoh masking citra sesungguhnya: citra AND mask A
masked_img = couple.copy()
masked_img[A == 0] = 0
show(masked_img, 'Image masking: couple.tiff AND mask A (bagian luar objek menjadi hitam)')

### Tugas 10 — Foto Malam Hari

Belum ada berkas khusus "foto malam" pada `week3`, sehingga dipakai `galaxy.jpg` — citra
langit malam (Deep Field) yang gelap dengan intensitas rata-rata sangat rendah namun
menyimpan detail objek redup (galaksi jauh). Metode yang dipilih: **Gamma Correction**
$\gamma = 2$, dibandingkan dengan **Logarithmic Brightness** dan kombinasi
gamma + contrast.

In [ ]:
# ---- Tugas 10: perbaikan foto malam hari (galaxy.jpg) ----
night = cv.imread(os.path.join(WEEK, 'galaxy.jpg'))
show(night, 'Foto malam asli - galaxy.jpg (rata-rata %.2f)' % night.mean())

gamma_night = gamma_correction(night, 2.0)
log_night = log_brightness(night)
gamma_contrast_night = contrast_transform(gamma_correction(night, 2.0), 20)

show(gamma_night, 'Gamma Correction gamma=2.0 (rata-rata %.2f)' % gamma_night.mean())
show(log_night, 'Logarithmic Brightness (rata-rata %.2f)' % log_night.mean())
show(gamma_contrast_night, 'Gamma 2.0 + Contrast C=20 (rata-rata %.2f)' % gamma_contrast_night.mean())

print('metode                rata-rata   std     p95   p5')
print('asli                  %8.2f %7.2f %5.0f %4.0f' % (night.mean(), night.std(), np.percentile(night, 95), np.percentile(night, 5)))
for nama, im in (('gamma 2.0', gamma_night), ('log brightness', log_night), ('gamma2 + C20', gamma_contrast_night)):
    print('%-20s  %8.2f %7.2f %5.0f %4.0f' % (nama, im.mean(), im.std(), np.percentile(im, 95), np.percentile(im, 5)))

### Tugas 11 — Perbaikan Kualitas `crayfish.jpg`

`crayfish.jpg` adalah foto makro lobster air tawar yang **agak gelap** (rata-rata
intensitas ≈ 123) dan memiliki **grain/noise** halus. Perbaikan dilakukan bertahap dengan
operasi dasar:

1. **Median blur 3×3** — mengurangi noise *salt-and-pepper*/grain tanpa menghaluskan tepi
   objek secara berlebihan (lebih baik daripada rata-rata/blur biasa untuk noise impulsif).
2. **Gamma Correction γ = 1.5** — menaikkan kecerahan area gelap (tubuh dan capit lobster)
   secara non-linier, sehingga detail bayangan ikut terangkat.
3. **Contrast C = 30** — meregangkan kembali intensitas di sekitar nilai tengah 128 agar
   citra tidak terlihat "pucat" setelah dinaikkan kecerahannya.

Parameter terbaik dipilih dari perbandingan kuantitatif (rata-rata, standar deviasi, dan
persentil 5/95) pada sel berikut.

In [ ]:
# ---- Tugas 11: perbaikan kualitas crayfish.jpg ----
cray = cv.imread(os.path.join(WEEK, 'crayfish.jpg'))
assert cray is not None, 'crayfish.jpg tidak ditemukan'

def ringkasan(nama, im):
    print('%-26s mean=%6.2f  std=%6.2f  p5=%3.0f  p95=%3.0f'
          % (nama, im.mean(), im.std(), np.percentile(im, 5), np.percentile(im, 95)))

ringkasan('asli', cray)
ringkasan('median blur 3x3', cv.medianBlur(cray, 3))
ringkasan('gamma 1.5', gamma_correction(cray, 1.5))
ringkasan('gamma 1.8', gamma_correction(cray, 1.8))
ringkasan('gamma 1.5 + C30', contrast_transform(gamma_correction(cray, 1.5), 30))
ringkasan('gamma 1.8 + C40', contrast_transform(gamma_correction(cray, 1.8), 40))

# Pipeline terpilih
denoise = cv.medianBlur(cray, 3)
bright = gamma_correction(denoise, 1.5)
final = contrast_transform(bright, 30)

show(cray, 'SEBELUM - crayfish.jpg (mean %.2f, std %.2f)' % (cray.mean(), cray.std()))
show(final, 'SESUDAH - median 3x3 + gamma 1.5 + contrast 30 (mean %.2f, std %.2f)' % (final.mean(), final.std()))
ringkasan('hasil akhir', final)

## Jawaban

### Jawaban Tugas 1 — Inverse Citra

Fungsi $g(x) = 255 - f(x)$ diterapkan langsung pada seluruh pixel dengan operasi NumPy
(`255 - img`), bukan `cv.bitwise_not`. Hasil pengamatan: kisaran nilai pixel tetap 0–255
(minimum/maksimum citra asli 0/255 tetap 0/255 pada citra negatif, hanya posisinya
bertukar), sehingga operasi ini tidak memerlukan truncate karena secara matematis tidak
mungkin keluar dari rentang. Bagian citra yang aslinya terang (cabai merah besar, lada
hijau) menjadi gelap, dan area gelap menjadi terang, sehingga terbentuk citra negatif yang
kontrasnya terbalik.

### Jawaban Tugas 2 — Transformasi Contrast

Rumus $F = \dfrac{259(C+255)}{255(259-C)}$ dengan $F$ bertipe **double** (di kode ditulis
`259.0`/`255.0` agar tidak terjadi pembagian integer). Untuk $C = 60$ diperoleh
$F \approx 1.42$, untuk $C = 120$ diperoleh $F \approx 3.67$, dan untuk $C = -60$
diperoleh $F \approx 0.59$. Semakin besar $C$ (positif) semakin lebar penyebaran intensitas
— dari keluaran program, standar deviasi intensitas naik dari 65.85 (asli) menjadi 93.65
pada $C = +60$ dan 110.06 pada $C = +120$, sedangkan pada $C = -60$ turun menjadi 40.89.
Artinya kontras memang menguat seiring bertambahnya $C$. Karena hasil $F(R-128)+128$ dapat melewati 0 atau 255, fungsi truncate wajib
dipakai dan pixel yang terpotong (misalnya area putih lada menjadi putih murni) kehilangan
informasi — operasi ini tidak dapat dibalik.

### Jawaban Tugas 3 — Transformasi Logarithmic Brightness

Dengan $s = c\log(1+r)$ dan $c = 255/\log(1+r_{max})$ (auto), bagian gray-level rendah
dipetakan ke rentang yang lebih lebar sehingga detail area gelap (bayangan antar-cabai)
terangkat dan rata-rata intensitas citra naik dari 110.73 (asli) menjadi 200.63. Efek ini
bersifat "menekan" nilai tinggi: pixel terang tidak banyak berubah, sedangkan pixel gelap
naik signifikan. Dengan $c$ kecil (misalnya 30) citra justru menjadi sangat gelap karena
seluruh nilai tertahan di bawah 255 — jadi pemilihan konstanta $c$ menentukan hasil akhir.

### Jawaban Tugas 4 — Grayscale (Averaging, Lightness, Luminance)

- **Averaging** $(R+G+B)/3$: paling sederhana, tetapi memberi bobot sama pada semua kanal
  sehingga objek berwarna terang seperti cabai merah dan lada hijau bisa memiliki nilai
  keabuan yang mirip walaupun persepsi matanya berbeda.
- **Lightness** $(\max+\min)/2$: mengabaikan kanal tengah sehingga nilai keabuannya berada
  di antara averaging dan luminance (rata-rata 115.29 pada `peppers.jpg`); hasilnya sering
  "cuci" pada area berwarna jenuh karena komponen tengah tidak diperhitungkan.
- **Luminance** $0.21R + 0.72G + 0.07B$: paling sesuai persepsi mata karena hijau
  (yang paling sensitif bagi mata) diberi bobot terbesar. Pada `peppers.jpg` rata-rata
  intensitas ketiga metode berbeda: averaging 110.40, lightness 115.29, dan luminance
  118.90 — luminance menghasilkan citra paling terang karena kanal hijau (lada hijau yang
  mendominasi citra) mendapat bobot 0.72.

Perhatikan bahwa OpenCV menyimpan citra sebagai BGR, sehingga indeks kanal harus dipetakan
dengan benar; kesalahan indeks akan menukar bobot R dan B dan menghasilkan grayscale yang
salah.

### Jawaban Tugas 5 — Selective Color

Mask merah dibentuk dengan syarat `R > G`, `R > B`, dan selisih `R - max(G,B) > 30`.
Pada `peppers.jpg` sekitar **44%** piksel memenuhi syarat tersebut (cabai merah besar dan
lada merah kecil di sudut). Pixel di dalam mask mempertahankan warna aslinya, sedangkan
sisanya diganti nilai luminance. Ambang `delta` berperan penting: `delta` terlalu kecil
membuat area oranye/kulit (yang juga didominasi R) ikut dipertahankan; `delta` terlalu besar
membuat sebagian cabai merah yang gelap (bayangan) berubah menjadi grayscale sehingga
mask berlubang-lubang.

### Jawaban Tugas 6 — Gamma Correction

Sesuai rumus modul $I' = 255 (I/255)^{1/\gamma}$: $\gamma = 1$ identik dengan citra asli,
$\gamma = 0.5$ membuat citra jauh lebih gelap (rata-rata turun dari 110.73 menjadi 64.65
karena eksponen menjadi 2), sedangkan $\gamma = 3$ (rata-rata 179.90) dan $\gamma = 6$
(rata-rata 209.42) membuat citra lebih terang (eksponen $1/3$ dan $1/6$). Karena `input()`
memblokir eksekusi dan tidak tersedia pada mode
non-interaktif Kaggle, notebook memakai `USE_INTERACTIVE_INPUT = False` dan nilai default
`gamma_value = 3`; blok `try/except ValueError` tetap dipertahankan sesuai petunjuk modul
sehingga bila pengguna mengetik bukan angka, program menampilkan `Error, not a number`
dan melanjutkan dengan nilai default.

### Jawaban Tugas 7 — Simulasi Image Depth

Dengan `level = 255/(2^bit_depth − 1)` dan `C' = round(C/level) × level`, jumlah nilai unik
pada citra hasil menyusut sesuai $2^{bit\_depth}$: bit_depth 1 → 2 nilai unik (0/255, citra
tampak seperti *stamp* hitam-putih), bit_depth 2 → 4 nilai, bit_depth 3 → 7 nilai,
bit_depth 4 → 15 nilai, bit_depth 5 → 29 nilai, bit_depth 6 → 58 nilai, dan bit_depth 7 →
117 nilai unik (jumlah nilai unik bisa sedikit lebih kecil dari $2^n$ karena tidak semua
level muncul pada citra tersebut). Efek yang terlihat adalah munculnya **banding
(posterisasi)**: gradasi halus pada permukaan lada berubah menjadi blok-blok warna datar.
Semakin kecil bit depth, semakin banyak informasi gradasi yang hilang. Citra 8-bit asli
setara dengan bit_depth 8 (256 warna).

### Jawaban Tugas 8 — Average Denoising dan PSNR

Hasil pengukuran nyata pada `galaxy.jpg` (PSNR dihitung terhadap citra asli):

| No | Jumlah Citra di Average | Image Hasil | Nilai PSNR (dB) |
|----|-------------------------|-------------|-----------------|
| 1 | 10 | average 10 citra | **31.6705** |
| 2 | 20 | average 20 citra | **32.6883** |
| 3 | 40 | average 40 citra | **33.2972** |
| 4 | 80 | average 80 citra | **33.6422** |
| 5 | 100 | average 100 citra | **33.7160** |

Sebagai pembanding, satu citra ber-noise tunggal hanya memiliki PSNR ≈ **24.90 dB**,
sehingga averaging sudah memberikan perbaikan > 6 dB bahkan pada 10 citra.

Peningkatan PSNR antar jumlah citra:

| Jumlah | PSNR (dB) | Selisih |
|--------|-----------|---------|
| 10 | 31.6705 | – |
| 20 | 32.6883 | +1.0178 |
| 40 | 33.2972 | +0.6089 |
| 80 | 33.6422 | +0.3450 |
| 100 | 33.7160 | +0.0738 |

**Apakah peningkatan jumlah citra selalu memberikan peningkatan PSNR yang signifikan?**
Tidak. PSNR selalu naik, tetapi kenaikannya **semakin mengecil** (bersifat logaritmik).
Secara teori, noise acak berkurang sebanding dengan $\sqrt{N}$, sehingga rata-rata dari 100
citra hanya menyisakan noise $1/\sqrt{100} = 1/10$ dari satu citra — perbaikan 10× pada
amplitudo noise setara kenaikan PSNR $20\log_{10}(10) = 20$ dB hanya jika noise benar-benar
independen dan tidak ada noise lain; pada data ini noise sudah didominasi komponen
terkuantitasi (JPEG) sehingga kenaikan terbatas.

**Pada jumlah citra berapa peningkatan mulai tidak terlalu signifikan?**
Mulai dari **40 citra ke atas**. Kenaikan 10→20 masih ≈ 1.02 dB, 20→40 turun menjadi
≈ 0.61 dB, 40→80 hanya ≈ 0.35 dB, dan 80→100 tinggal ≈ 0.07 dB. Jadi setelah sekitar
40 citra, tambahan citra tidak lagi sebanding dengan biaya komputasi/penyimpanan.

**Kesimpulan:** Averaging adalah metode denoising yang efektif untuk noise acak
(uncorrelated) karena sinyal yang konsisten antar citra saling menguatkan sementara noise
saling meniadakan. Namun manfaatnya mengikuti hukum *diminishing returns*: peningkatan PSNR
sebanding dengan $\log(\sqrt{N})$, sehingga menambah citra dari 80 ke 100 hampir tidak
terasa secara visual. Jumlah citra yang *reasonable* untuk praktik adalah 20–40; lebih dari
itu hasilnya jenuh (mendekati batas noise sisa yang tidak lagi acak). Perhatikan juga bahwa
averaging hanya menghapus noise yang **berbeda antar citra**; jika noise yang sama muncul
pada semua citra, ia akan tetap ada pada hasil rata-rata.

### Jawaban Tugas 9 — Image Masking (Analisa Operator)

Citra biner **A** adalah hasil threshold Otsu manual pada `couple.tiff` (nilai threshold =
142; objek terang — kemeja, celana, gaun — menjadi putih seluas ≈ 77% citra, sedangkan latar
jalan/taman menjadi hitam), dan **B** adalah A yang digeser 40 piksel ke kanan.

| No. | Operator | Rumus | Hasil pengamatan | Persentase piksel putih |
|-----|----------|-------|------------------|--------------------------|
| 1 | NOT (komplemen) | $255 - A$ | Seluruh citra terbalik: objek yang tadinya putih menjadi hitam, latar menjadi putih. Ini menghasilkan *silhouette* objek. | 22.78% |
| 2 | OR (atau) | $A \mid B$ | Gabungan A dan B — area putih bertambah karena bagian yang hanya aktif di A atau hanya di B ikut menyala; bentuk objek tampak "melebar" ke kanan. | 90.83% |
| 3 | AND (dan) | $A \mathbin{\&} B$ | Hanya irisan A dan B yang putih — objek menyusut menjadi bentuk yang lebih sempit (bagian yang tidak beririsan karena pergeseran hilang). | 63.61% |
| 4 | NAND (not and) | $255 - (A \mathbin{\&} B)$ | Kebalikan dari AND: yang putih adalah semua area **kecuali** irisan; area irisan menjadi hitam. | 36.39% |
| 5 | XOR (exclusive or) | $A \mathbin{\hat{}} B$ | Aktif hanya jika salah satu (bukan keduanya) aktif — hasilnya berupa pita tegak selebar 40 piksel di tepi kiri dan kanan objek (bagian yang tergeser), sedangkan daerah irisan justru menjadi hitam. | 27.22% |

**Analisa:** Semua operator bekerja per-pixel pada citra biner, sehingga hasilnya ditentukan
sepenuhnya oleh tabel kebenaran logika. NOT membalik foreground/background, OR bersifat
"union" (selalu menambah area putih), AND bersifat "intersection" (selalu mempersempit),
NAND adalah komplemen dari AND, dan XOR menandai **perbedaan** antara kedua citra sehingga
paling berguna untuk mendeteksi pergeseran/perubahan antar dua citra. Pada pengolahan citra
nyata, operator AND dipakai untuk **masking**: citra asli `couple.tiff` di-AND-kan dengan
mask A sehingga hanya area objek yang tersisa dan latar menjadi hitam. Kombinasi A dan B
yang berbeda (bukan hanya hasil pergeseran) akan memberikan bentuk keluaran yang berbeda,
tetapi sifat aljabar operatornya tetap sama.

### Jawaban Tugas 10 — Foto Malam Hari

Foto malam yang dipakai adalah `galaxy.jpg` (citra langit malam, rata-rata intensitas
hanya ≈ 19). Metode yang dipilih: **Gamma Correction dengan $\gamma = 2$**.

- **Alasan pemilihan:** gamma correction bekerja secara non-linier — kenaikannya besar pada
  nilai pixel rendah (area gelap) dan kecil pada nilai tinggi, sehingga detail objek redup
  terangkat tanpa membuat area terang (bintang terang) jenuh (blown out). Rata-rata
  intensitas naik dari ≈ 19 menjadi ≈ 62, dan persentil 5 naik dari 4 menjadi ≈ 31 sehingga
  area latar yang tadinya hampir hitam mulai memperlihatkan tekstur.
- **Perbandingan:** Logarithmic Brightness menghasilkan citra jauh lebih terang
  (rata-rata ≈ 122) tetapi membuat latar langit menjadi keabuan sehingga kontras antara
  bintang dan latar berkurang — langit malam "hilang" kesannya. Kombinasi gamma 2 + contrast
  20 menggelapkan kembali latar (rata-rata ≈ 51) dan menaikkan standar deviasi sehingga
  bintang tampak lebih "pop" — alternatif yang baik bila tujuannya menonjolkan objek.
- **Resiko:** gamma correction **ikut memperkuat noise** pada area gelap (noise ikut
  dipangkatkan ke $1/\gamma$), sehingga foto malam ber-noise tinggi akan tampak lebih
  *grainy*. Selain itu, operasi ini juga *truncate*: pixel yang sudah mendekati 255 akan
  jenuh dan kehilangan detail (halo di sekitar bintang terang). Karena itu untuk foto malam
  nyata sebaiknya dilakukan denoising terlebih dahulu, atau gunakan $\gamma$ kecil (1.5–2)
  agar noise tidak ikut meledak.

### Jawaban Tugas 11 — Perbaikan Kualitas `crayfish.jpg`

**Metode yang dipilih:** pipeline tiga tahap operasi dasar —
(1) `cv.medianBlur` 3×3, (2) Gamma Correction $\gamma = 1.5$, (3) Contrast $C = 30$.

**Alasan pemilihan:** citra asli memiliki rata-rata intensitas 122.72 dengan detail yang
terlalu gelap pada tubuh/capit lobster dan grain halus pada latar berpasir.
- Median blur 3×3 dipilih (bukan rata-rata/blur Gaussian) karena menghilangkan noise
  impulsif tanpa mengaburkan tepi keras capit dan kaki lobster; standar deviasi turun
  sedikit (44.65 → 43.50) yang menandakan noise berkurang sementara rata-rata hampir
  tidak berubah (122.72 → 122.40).
- Gamma $\gamma = 1.5$ menaikkan kecerahan secara non-linier: area gelap terangkat lebih
  banyak daripada area terang, sehingga detail lobster muncul tanpa membuat latar menjadi
  putih jenuh (rata-rata 122.72 → 153.14, persentil 5 naik dari 34 menjadi 66).
- Contrast $C = 30$ ($F = \frac{259 \cdot 285}{255 \cdot 229} \approx 1.26$) meregangkan
  kembali intensitas di sekitar 128 agar citra tidak terlihat pucat setelah dinaikkan
  kecerahannya; pada hasil akhir standar deviasi naik (43.50 → 50.16) sehingga citra
  tampak lebih tajam dan "hidup".

**Menentukan parameter terbaik:** dilakukan dengan membandingkan statistik (mean, std,
persentil 5 dan 95) pada sel eksperimen. `gamma = 1.5` dipilih karena `gamma = 1.8`
(rata-rata 165.67, persentil 95 naik ke 210) sudah mulai menjenuhkan bagian terang,
sedangkan `gamma = 1.5` (rata-rata 153.14) cukup mengangkat detail gelap. `C = 30` dipilih
karena kombinasi `gamma 1.8 + C 40` sudah menimbulkan *clipping* berlebihan pada sorotan air
(persentil 95 = 240, mendekati putih murni) dan beresiko halo di tepi capit.

**Hasil before–after:** ditampilkan pada sel Tugas 11 — sebelum (gelap, agak *grainy*) dan
sesudah (lebih terang, detail capit/kaki lebih jelas, kontras lebih tegas, noise latar
berkurang). Perlu dicatat bahwa seluruh operasi ini memakai truncate, sehingga sebagian
informasi pixel yang terpotong tidak dapat dikembalikan.

### Kesimpulan Modul 3

1. Operasi titik (*point operations*) — inverse, linier brightness, contrast, log, gamma,
   dan kuantisasi bit depth — bekerja pada setiap pixel secara independen dan seluruhnya
   memerlukan **truncate** (kecuali inverse yang secara matematis selalu aman) agar nilai
   tetap pada rentang 0–255. Konsekuensinya, semua operasi tersebut **tidak dapat
   direverse** dengan sempurna.
2. Perbedaan utama antar transformasi kecerahan terletak pada bentuk kurvanya: linier
   brightness menggeser seluruh histogram, contrast meregangkannya dari titik tengah 128,
   sedangkan log dan gamma memetakan ulang secara non-linier sehingga sangat efektif untuk
   menyeimbangkan area gelap/terang yang ekstrem.
3. Grayscale dapat dihitung dengan beberapa pendekatan; luminance paling mendekati
   persepsi manusia karena memberi bobot terbesar pada kanal hijau.
4. Averaging adalah denoising efektif untuk noise acak, dengan kenaikan PSNR yang
   mengikuti pola *diminishing returns* ($\propto \log\sqrt{N}$); 20–40 citra sudah cukup.
5. Operator logika (NOT/OR/AND/NAND/XOR) membentuk dasar image masking; AND adalah operator
   yang dipakai untuk memotong citra dengan mask, sedangkan XOR paling berguna untuk
   mendeteksi perbedaan antar dua citra biner.